# Datetime operations
## Refreshing basic datetime operations

In [12]:
import pandas as pd

df = pd.DataFrame({
    "event": ["Orientation", "Lecture", "Exam", "Workshop", "Holiday"],
    "date": ["2024-08-20", "2024-08-21", "2024-8-22", "2024-08-24", "invalid_date"]
})

print(df)


         event          date
0  Orientation    2024-08-20
1      Lecture    2024-08-21
2         Exam     2024-8-22
3     Workshop    2024-08-24
4      Holiday  invalid_date


### Convert to datetime date type

In [13]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print(df)


         event       date
0  Orientation 2024-08-20
1      Lecture 2024-08-21
2         Exam 2024-08-22
3     Workshop 2024-08-24
4      Holiday        NaT


### Use `.dt` Accessor and create new columns

In [15]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["weekday"] = df["date"].dt.weekday  # Monday=0, Sunday=6
df['is_weekend'] = df['date'].dt.weekday >=5

print(df)


         event       date    year  month   day  weekday  is_weekend
0  Orientation 2024-08-20  2024.0    8.0  20.0      1.0       False
1      Lecture 2024-08-21  2024.0    8.0  21.0      2.0       False
2         Exam 2024-08-22  2024.0    8.0  22.0      3.0       False
3     Workshop 2024-08-24  2024.0    8.0  24.0      5.0        True
4      Holiday        NaT     NaN    NaN   NaN      NaN       False


## Period date type
A Period represents a span of time, not a single point.
### Create Monthly Periods

In [16]:
import pandas as pd

# Create a monthly period
p = pd.Period("2024-08", freq="M")

print(p)
print(type(p))


2024-08
<class 'pandas._libs.tslibs.period.Period'>


### Access attributes

In [17]:
print(p.year)
print(p.month)
print(p.start_time)
print(p.end_time)


2024
8
2024-08-01 00:00:00
2024-08-31 23:59:59.999999999


### Period arithmetic

In [19]:
print(p + 1)   # next month
print(p - 2)   # two months earlier
print((p+1).start_time)


2024-09
2024-06
2024-09-01 00:00:00


### Create period Series
Convert datetime to period Series, you can use using `.dt.to_period("M")`

In [20]:
df = pd.DataFrame({
    "month": ["2024-06", "2024-07", "2024-08"]
})

df["month"] = pd.to_datetime(df["month"]).dt.to_period("M")

print(df)


     month
0  2024-06
1  2024-07
2  2024-08


### Group by Period (time aggregation)

In [45]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "date": pd.date_range(start="2024-08-01", periods=10, freq="3B"),
    "sales": [10, 15, 12, 20, 18, 25, 30, 28, 22, 19]
})

df


,date,sales
0,2024-08-01,10
1,2024-08-06,15
2,2024-08-09,12
3,2024-08-14,20
4,2024-08-19,18
5,2024-08-22,25
6,2024-08-27,30
7,2024-08-30,28
8,2024-09-04,22
9,2024-09-09,19


In [46]:
# convert to monthly period

df["month"] = df["date"].dt.to_period("M")

df


,date,sales,month
0,2024-08-01,10,2024-08
1,2024-08-06,15,2024-08
2,2024-08-09,12,2024-08
3,2024-08-14,20,2024-08
4,2024-08-19,18,2024-08
5,2024-08-22,25,2024-08
6,2024-08-27,30,2024-08
7,2024-08-30,28,2024-08
8,2024-09-04,22,2024-09
9,2024-09-09,19,2024-09


In [47]:
# group by month

monthly_sales = df.groupby("month")["sales"].sum()

monthly_sales


month
2024-08    158
2024-09     41
Freq: M, Name: sales, dtype: int64

## Filtering by datetime

example: create class schedule for EST 389


In [63]:
import pandas as pd

# Create date range (daily first)
dates = pd.date_range(start="2026-01-26", end="2026-05-06", freq="D")

# Remove spring break
dates = dates[~dates.to_series().between("2026-03-16", "2026-03-22")]

# Filter Mondays (0) and Wednesdays (2)
mw_dates = dates[dates.weekday.isin([0, 2])]

# Add time (9:30 AM)
mw_start_times = mw_dates + pd.Timedelta(hours=9, minutes=30)
mw_end_times = mw_start_times + pd.Timedelta(hours=1, minutes=20)

# Create DataFrame
df = pd.DataFrame({"class_time": mw_dates,
                   "start_time": mw_start_times,
                   "end_time": mw_end_times})

df["class_period"] = pd.IntervalIndex.from_arrays(df["start_time"], df["end_time"], closed="both")
df['week'] = df['class_time'].dt.to_period("W").rank(method='dense').astype(int)

df.head()


,class_time,start_time,end_time,class_period,week
0,2026-01-26,2026-01-26 09:30:00,2026-01-26 10:50:00,"[2026-01-26 09:30:00, 2026-01-26 10:50:00]",1
1,2026-01-28,2026-01-28 09:30:00,2026-01-28 10:50:00,"[2026-01-28 09:30:00, 2026-01-28 10:50:00]",1
2,2026-02-02,2026-02-02 09:30:00,2026-02-02 10:50:00,"[2026-02-02 09:30:00, 2026-02-02 10:50:00]",2
3,2026-02-04,2026-02-04 09:30:00,2026-02-04 10:50:00,"[2026-02-04 09:30:00, 2026-02-04 10:50:00]",2
4,2026-02-09,2026-02-09 09:30:00,2026-02-09 10:50:00,"[2026-02-09 09:30:00, 2026-02-09 10:50:00]",3


In [64]:
# Filter a Specific Time Period (March)

march_classes = df[
    (df["class_time"] >= "2026-03-01") &
    (df["class_time"] <= "2026-03-31")
]

march_classes

,class_time,start_time,end_time,class_period,week
10,2026-03-02,2026-03-02 09:30:00,2026-03-02 10:50:00,"[2026-03-02 09:30:00, 2026-03-02 10:50:00]",6
11,2026-03-04,2026-03-04 09:30:00,2026-03-04 10:50:00,"[2026-03-04 09:30:00, 2026-03-04 10:50:00]",6
12,2026-03-09,2026-03-09 09:30:00,2026-03-09 10:50:00,"[2026-03-09 09:30:00, 2026-03-09 10:50:00]",7
13,2026-03-11,2026-03-11 09:30:00,2026-03-11 10:50:00,"[2026-03-11 09:30:00, 2026-03-11 10:50:00]",7
14,2026-03-23,2026-03-23 09:30:00,2026-03-23 10:50:00,"[2026-03-23 09:30:00, 2026-03-23 10:50:00]",8
15,2026-03-25,2026-03-25 09:30:00,2026-03-25 10:50:00,"[2026-03-25 09:30:00, 2026-03-25 10:50:00]",8
16,2026-03-30,2026-03-30 09:30:00,2026-03-30 10:50:00,"[2026-03-30 09:30:00, 2026-03-30 10:50:00]",9


In [65]:
# Alternative way to filter March class

df[df["class_time"].dt.month == 3]


,class_time,start_time,end_time,class_period,week
10,2026-03-02,2026-03-02 09:30:00,2026-03-02 10:50:00,"[2026-03-02 09:30:00, 2026-03-02 10:50:00]",6
11,2026-03-04,2026-03-04 09:30:00,2026-03-04 10:50:00,"[2026-03-04 09:30:00, 2026-03-04 10:50:00]",6
12,2026-03-09,2026-03-09 09:30:00,2026-03-09 10:50:00,"[2026-03-09 09:30:00, 2026-03-09 10:50:00]",7
13,2026-03-11,2026-03-11 09:30:00,2026-03-11 10:50:00,"[2026-03-11 09:30:00, 2026-03-11 10:50:00]",7
14,2026-03-23,2026-03-23 09:30:00,2026-03-23 10:50:00,"[2026-03-23 09:30:00, 2026-03-23 10:50:00]",8
15,2026-03-25,2026-03-25 09:30:00,2026-03-25 10:50:00,"[2026-03-25 09:30:00, 2026-03-25 10:50:00]",8
16,2026-03-30,2026-03-30 09:30:00,2026-03-30 10:50:00,"[2026-03-30 09:30:00, 2026-03-30 10:50:00]",9


In [67]:
# Filter out Mondays, the days where we will have "Data Science Project of the Day"

mondays = df[df["class_time"].dt.weekday == 0]

mondays.head()


,class_time,start_time,end_time,class_period,week
0,2026-01-26,2026-01-26 09:30:00,2026-01-26 10:50:00,"[2026-01-26 09:30:00, 2026-01-26 10:50:00]",1
2,2026-02-02,2026-02-02 09:30:00,2026-02-02 10:50:00,"[2026-02-02 09:30:00, 2026-02-02 10:50:00]",2
4,2026-02-09,2026-02-09 09:30:00,2026-02-09 10:50:00,"[2026-02-09 09:30:00, 2026-02-09 10:50:00]",3
6,2026-02-16,2026-02-16 09:30:00,2026-02-16 10:50:00,"[2026-02-16 09:30:00, 2026-02-16 10:50:00]",4
8,2026-02-23,2026-02-23 09:30:00,2026-02-23 10:50:00,"[2026-02-23 09:30:00, 2026-02-23 10:50:00]",5


In [69]:
# Create a new column to indicate whether we will have data science project of the day
df['have_ds_project'] = df["class_time"].dt.weekday == 0

df

,class_time,start_time,end_time,class_period,week,have_ds_project
0,2026-01-26,2026-01-26 09:30:00,2026-01-26 10:50:00,"[2026-01-26 09:30:00, 2026-01-26 10:50:00]",1,True
1,2026-01-28,2026-01-28 09:30:00,2026-01-28 10:50:00,"[2026-01-28 09:30:00, 2026-01-28 10:50:00]",1,False
2,2026-02-02,2026-02-02 09:30:00,2026-02-02 10:50:00,"[2026-02-02 09:30:00, 2026-02-02 10:50:00]",2,True
3,2026-02-04,2026-02-04 09:30:00,2026-02-04 10:50:00,"[2026-02-04 09:30:00, 2026-02-04 10:50:00]",2,False
4,2026-02-09,2026-02-09 09:30:00,2026-02-09 10:50:00,"[2026-02-09 09:30:00, 2026-02-09 10:50:00]",3,True
5,2026-02-11,2026-02-11 09:30:00,2026-02-11 10:50:00,"[2026-02-11 09:30:00, 2026-02-11 10:50:00]",3,False
6,2026-02-16,2026-02-16 09:30:00,2026-02-16 10:50:00,"[2026-02-16 09:30:00, 2026-02-16 10:50:00]",4,True
7,2026-02-18,2026-02-18 09:30:00,2026-02-18 10:50:00,"[2026-02-18 09:30:00, 2026-02-18 10:50:00]",4,False
8,2026-02-23,2026-02-23 09:30:00,2026-02-23 10:50:00,"[2026-02-23 09:30:00, 2026-02-23 10:50:00]",5,True
9,2026-02-25,2026-02-25 09:30:00,2026-02-25 10:50:00,"[2026-02-25 09:30:00, 2026-02-25 10:50:00]",5,False


## Compute time differences and sort by time

In [81]:
import pandas as pd

df = pd.DataFrame({
    "class": ["EST389", "EST532", "CSE232", "AMP314"],
    "start_time": [
        "2026-02-09 09:30",
        "2026-02-02 09:30",
        "2026-02-11 09:30",
        "2026-02-04 09:30"
    ],
    "end_time": [
        "2026-02-09 10:40",
        "2026-02-02 10:45",
        "2026-02-11 11:00",
        "2026-02-04 10:50"
    ]
})

# Convert to datetime
df["start_time"] = pd.to_datetime(df["start_time"])
df["end_time"] = pd.to_datetime(df["end_time"])

df


,class,start_time,end_time
0,EST389,2026-02-09 09:30:00,2026-02-09 10:40:00
1,EST532,2026-02-02 09:30:00,2026-02-02 10:45:00
2,CSE232,2026-02-11 09:30:00,2026-02-11 11:00:00
3,AMP314,2026-02-04 09:30:00,2026-02-04 10:50:00


In [82]:
# compute duration

df["duration"] = df["end_time"] - df["start_time"]

df


,class,start_time,end_time,duration
0,EST389,2026-02-09 09:30:00,2026-02-09 10:40:00,0 days 01:10:00
1,EST532,2026-02-02 09:30:00,2026-02-02 10:45:00,0 days 01:15:00
2,CSE232,2026-02-11 09:30:00,2026-02-11 11:00:00,0 days 01:30:00
3,AMP314,2026-02-04 09:30:00,2026-02-04 10:50:00,0 days 01:20:00


In [83]:
# convert durations to minutes

df["duration_min"] = df["duration"].dt.total_seconds() / 60

df


,class,start_time,end_time,duration,duration_min
0,EST389,2026-02-09 09:30:00,2026-02-09 10:40:00,0 days 01:10:00,70.0
1,EST532,2026-02-02 09:30:00,2026-02-02 10:45:00,0 days 01:15:00,75.0
2,CSE232,2026-02-11 09:30:00,2026-02-11 11:00:00,0 days 01:30:00,90.0
3,AMP314,2026-02-04 09:30:00,2026-02-04 10:50:00,0 days 01:20:00,80.0


In [84]:
# Compute Time Between Classes

df["gap_since_last"] = df["start_time"].diff()

df


,class,start_time,end_time,duration,duration_min,gap_since_last
0,EST389,2026-02-09 09:30:00,2026-02-09 10:40:00,0 days 01:10:00,70.0,NaT
1,EST532,2026-02-02 09:30:00,2026-02-02 10:45:00,0 days 01:15:00,75.0,-7 days
2,CSE232,2026-02-11 09:30:00,2026-02-11 11:00:00,0 days 01:30:00,90.0,9 days
3,AMP314,2026-02-04 09:30:00,2026-02-04 10:50:00,0 days 01:20:00,80.0,-7 days


In [86]:
# See that gap_between_last has negative numbers, so it is better we sort by time first

df = df.sort_values(by="start_time", ascending=True)

df["gap_since_last"] = df["start_time"].diff()

df


,class,start_time,end_time,duration,duration_min,gap_since_last
1,EST532,2026-02-02 09:30:00,2026-02-02 10:45:00,0 days 01:15:00,75.0,NaT
3,AMP314,2026-02-04 09:30:00,2026-02-04 10:50:00,0 days 01:20:00,80.0,2 days
0,EST389,2026-02-09 09:30:00,2026-02-09 10:40:00,0 days 01:10:00,70.0,5 days
2,CSE232,2026-02-11 09:30:00,2026-02-11 11:00:00,0 days 01:30:00,90.0,2 days


In [94]:
# group by the day of the week

df['weekday'] = df['start_time'].dt.day_name()

df.groupby("weekday")['class'].count().reset_index().rename(columns={'class': 'class_count'})


,weekday,class_count
0,Monday,2
1,Wednesday,2
